# Lesson 5: Sentiment Analysis on Toponym Sentences

## Overview

This lesson will cover two sentiment analysis methods:
- Using the **NLTK** library's VADER sentiment analysis tool.
- Using **Hugging Face's RoBERTa** model for sentiment analysis.

We will compare how these two tools perform on sentences containing toponyms extracted from the `jmu_reddit_geoparsed_long.pickle` file, and we will store the results in a **pandas** DataFrame for further analysis. The key goal is to understand how different tools analyze sentiment, identify their limitations, and explore why their outputs might differ. Unlike the `geoparser` you will run the RoBERTa model to get your own results for the final project.

---

## 1. Sentiment Analysis with NLTK (VADER)

### 1.1 Overview
VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analysis tool specifically attuned to sentiments expressed in social media. It was largely trained on Twitter data and evaluates sentiment at the word level. This makes it fast, but it has notable limitations.

The sentiment analyzer is easy to use. Once the libraries are loaded calling the `sia.polarity_scores` function on any text performs the analysis. For example:

```python
sia.polarity_scores('I love going to Buc-ees, but I do not love the gas prices there.')
```

Output:
```
{'neg': 0.0, 'neu': 0.543, 'pos': 0.457, 'compound': 0.8555}
```

The four scores are:

| Score | Range | Meaning |
|---|---|---|
| `neg` | 0.0 – 1.0 | Proportion of text that is negative |
| `neu` | 0.0 – 1.0 | Proportion of text that is neutral |
| `pos` | 0.0 – 1.0 | Proportion of text that is positive |
| `compound` | -1.0 – +1.0 | Overall sentiment: < -0.05 = negative, > 0.05 = positive, in between = neutral |

### 1.2 Evaluating Positive Scores

In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize the VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()
sia.polarity_scores("I love going to Buc-ees, but I do not love the gas prices there.")

{'neg': 0.0, 'neu': 0.543, 'pos': 0.457, 'compound': 0.8555}

> 💡 **Reflection:** The positive score is `0.45` and the neutral score is `0.54`. 
> - What parts of the sentence might the algorithm be seeing as positive, and what parts as neutral? 
> The overall compound score is `0.86`. 
> - Do you agree with this assessment? Is it too positive, or too negative?

### 1.3 Evaluating Negative Scores

The sentence below shares the same negation principle as the above sentence where `not` is being used to create the opposite of a particular feeling.

In [2]:
sia.polarity_scores('UVA is not the best university!')

{'neg': 0.423, 'neu': 0.577, 'pos': 0.0, 'compound': -0.5661}

> 💡 **Reflection:** What do you make of this assessment? Why is this score so much more negative than the above sentence which was overwhelmingly positive?

### ✍️ 1.4 Critical Activity - Making Emotions

For the next activity, you are going to try to push the limits of the tokenizer. For each challenge, think of a sentence that will get the scores you want, even if those scores don't make sense.

#### Challenge 1: Most Goodest Vibes

Create a sentence with a compound polarity score of 1.0.

In [9]:
sia.polarity_scores('')

{'neg': 0.0, 'neu': 0.0, 'pos': 0.0, 'compound': 0.0}

#### Challenge 2: Most Baddest Vibes

 Create a sentence with a compound polarity score of -1.0, but keep it PG-13 so you can share with the class!

In [ ]:
sia.polarity_scores('')

#### Challenge 3: Most Strangest Vibes

Create a sentence with either a positive or negative compound score, but that means the exact opposite of what it says.

In [ ]:
sia.polarity_scores('')

> 💡 **Reflection:** What were some ways you manipulated the tokenizer to get the result you wanted? How does this show the limits of what the tokenizer is capable of?

### 1.5 Run and Evaluate VADER

While VADER is relatively quick and easy to implement, some of the quirks of how it evaluates sentiment become visible when applied to a large dataset. Having been trained on Twitter it excels at passages that lack nuance, but has a much harder time with complicated human emotions.

The code below runs the analysis on all lines and produces a table with outputs.

In [3]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize the VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

df_reddit_geoparsed_long = pd.read_csv("../data/JMU/JMU_geoparsed_long_backup.csv")
# Perform sentiment analysis on each sentence and store the compound score in the DataFrame
df_reddit_geoparsed_long['vader_sentiment'] = df_reddit_geoparsed_long['sentences'].apply(lambda x: sia.polarity_scores(x)['compound'])
(df_reddit_geoparsed_long[["sentences", "vader_sentiment"]]
    .sample(5, random_state=6)
    .style
    .set_properties(subset=['sentences'], **{'white-space': 'normal', 'width': '600px'})
    .set_properties(subset=['vader_sentiment'], **{'width': '120px', 'text-align': 'center'})
    .format({'vader_sentiment': '{:.3f}'})
)

,sentences,vader_sentiment
60,I lived in Logan my first year and I genuinely don’t think I could have been happier in another dorm.,0.527
1129,A Harrisonburg LARPing group meets every Saturday in Hillandale Park.,0.000
102,Thank you from an Alum (13/14) turned Harrisonburg local.,0.361
376,We are willing to ship the supplies to anywhere in VIRGINIA.,0.000
1558,There is also so much to do right outside of Harrisonburg.,0.000


The compound score ranges from -1 to 1. When a passage is very negative it gets a -1 and when it is possitive it gets a 1. Read through the passages above and try to figure out why these passages received the sentiments they did.

> 💡 **Critical Reflection:** How effective is the VADER tokenizer in dealing with sentiments?
> - Look at the sample output above. Are there any sentences where the score surprises you?
> - VADER was trained on Twitter data. Does that seem to match the tone of Reddit posts?
> - What kinds of sentences do you think would consistently fool a rule-based tool like VADER?


## Lesson Summary

Here is what you covered in this lesson:

### Part 1: Rule-Based Sentiment Analysis with VADER
- **VADER** — a rule-based tool trained on social media text; scores sentiment at the word level
- **`sia.polarity_scores(text)`** — returns `neg`, `neu`, `pos`, and `compound` scores for any string
- **`compound`** — the overall sentiment score, ranging from -1.0 (very negative) to +1.0 (very positive)
- **Limitations** — VADER struggles with negation, sarcasm, and complex emotional nuance; it treats every word independently

### Key pandas skills used
- **`.apply(lambda x: ...)`** — run a function on every row of a column
- **`.sample(n, random_state=)`** — draw a reproducible random sample for inspection
- **`.style`** — apply display formatting to a DataFrame without changing the underlying data

---

In the next lesson you will run a transformer-based model (RoBERTa) on the same data and compare its results to VADER.

➡️ **Next:** [Lesson 5.2 — Sentiment Analysis with RoBERTa](lesson_5_2_roberta_sentiment.ipynb)
